# Sesión 1 · Mini-Colab: ¿cuánta información hay en un texto y en un LLM?

**Teoría de la Información · Prof. Víctor Adrián Sosa Hernández · 15 de octubre de 2026**

Hoy vas a medir entropías de datos reales:

1. **Español escrito:** las letras de *El Quijote*.
2. **Un modelo de lenguaje real:** los 134.5 millones de pesos de *SmolLM2-135M*.

Recuerda la definición de **entropía** (ecuación 2 de la presentación):
$$H(X) = -\sum_x p(x)\log_2 p(x) \tag{2}$$

| Parte | Qué significa |
|---|---|
| $X$ | variable aleatoria: el símbolo que produce la fuente (una letra, un campo de un peso…) |
| $x$ | cada valor posible de $X$ |
| $p(x)$ | probabilidad de que $X$ valga $x$; aquí se **estima** como (veces que aparece $x$) / (total) |
| $-\log_2 p(x)$ | sorpresa de ver $x$, en bits (ecuación 1) |
| $\sum_x p(x)(\cdots)$ | promedio ponderado: por eso la entropía es la **sorpresa promedio** (*expected surprisal*) |

> Ejecuta las celdas en orden (`Shift + Enter`). Las celdas marcadas con ✍️ son para ti.
> **Tip:** si la red está lenta, ejecuta primero la celda **2.1** (descarga) y mientras haz la parte 1.

In [ ]:
import numpy as np, collections, unicodedata, re, json, struct, zlib, os, urllib.request
import matplotlib.pyplot as plt

def entropia(conteos):
    # Entropía en bits de una distribución dada por conteos (array o lista)
    c = np.asarray(conteos, dtype=float)
    p = c[c > 0] / c.sum()
    return -(p * np.log2(p)).sum()

# Prueba rápida: moneda justa, moneda cargada, dado
print(entropia([1, 1]), entropia([9, 1]), entropia([1]*6))

## Parte 1 · Entropía del español

### 1.1 Descargar y limpiar el texto
Dejamos sólo minúsculas `a–z`, la `ñ` y el espacio (quitamos acentos).

In [ ]:
URL_QUIJOTE = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
if not os.path.exists("quijote.txt"):
    urllib.request.urlretrieve(URL_QUIJOTE, "quijote.txt")

t = open("quijote.txt", encoding="utf-8").read()
t = t[t.find("En un lugar de la Mancha"): t.find("*** END")].lower()
# quitar acentos pero conservar la ñ
t = unicodedata.normalize("NFD", t)
t = "".join(c for c in t if unicodedata.category(c) != "Mn" or c == "̃")
t = unicodedata.normalize("NFC", t)
t = re.sub("[^a-zñ ]+", " ", t)
t = re.sub(" +", " ", t)
print(len(t), "caracteres")
print(t[:300])

### 1.2 Entropía de una letra

In [ ]:
c1 = collections.Counter(t)
H1 = entropia(list(c1.values()))
print(f"Alfabeto: {len(c1)} símbolos  ->  máximo posible log2({len(c1)}) = {np.log2(len(c1)):.3f} bits")
print(f"H(letra) = {H1:.3f} bits")

top = c1.most_common()
plt.figure(figsize=(10, 3))
plt.bar([repr(k)[1:-1] if k != ' ' else '␣' for k, _ in top], [v / len(t) for _, v in top])
plt.title("Frecuencia de símbolos en El Quijote"); plt.ylabel("p"); plt.show()

### 1.3 ✍️ Entropía condicionada a la letra anterior

Usa la regla de la cadena (ecuación 8 de la presentación), despejada:
$$H(X_2 \mid X_1) = H(X_1, X_2) - H(X_1) \tag{8}$$

| Parte | Qué significa |
|---|---|
| $X_1, X_2$ | dos letras consecutivas del texto |
| $H(X_1, X_2)$ | entropía conjunta del par (ecuación 6): se calcula contando **parejas** de letras |
| $H(X_1)$ | entropía de una letra sola (la de 1.2) |
| $H(X_2 \mid X_1)$ | incertidumbre que queda sobre una letra cuando ya se conoce la anterior (ecuación 7) |

Completa la celda: cuenta los **pares** de letras consecutivas y calcula $H(X_1, X_2)$.

In [ ]:
pares = collections.Counter(zip(t, t[1:]))     # conteo de bigramas
H12 = entropia(list(pares.values()))            # H(X1, X2)
print(f"H(X1, X2)   = {H12:.3f} bits")
print(f"H(X2 | X1)  = {H12 - H1:.3f} bits   (vs H(X) = {H1:.3f})")

**Pregunta ✍️:** ¿por qué baja la entropía al conocer la letra anterior? ¿Qué crees que pase si condicionas
en las **dos** letras anteriores? Pruébalo (pista: tripletas `zip(t, t[1:], t[2:])`).

*Tu respuesta:*

In [ ]:
# ✍️ Tu código aquí: H(X3 | X1, X2) = H(X1, X2, X3) - H(X1, X2)

## Parte 2 · ¿Cuánta información hay en un peso de un LLM?

*SmolLM2-135M* guarda cada peso en formato `bfloat16` (16 bits):

```
 ┌─┬──────────┬─────────┐
 │s│ exponente │ mantisa │    1 + 8 + 7 = 16 bits
 └─┴──────────┴─────────┘
```

El archivo `.safetensors` es muy simple: 8 bytes con el tamaño del encabezado, un encabezado JSON, y después
**todos los pesos crudos, uno tras otro**. No necesitamos PyTorch: lo leemos como enteros de 16 bits.

### 2.1 Descargar el modelo (270 MB)

In [ ]:
URL_MODELO = "https://huggingface.co/HuggingFaceTB/SmolLM2-135M/resolve/main/model.safetensors"
ARCHIVO = "smol.safetensors"
if not os.path.exists(ARCHIVO):
    urllib.request.urlretrieve(URL_MODELO, ARCHIVO)
print(os.path.getsize(ARCHIVO) / 1e6, "MB")

### 2.2 Leer el encabezado y los pesos como bits

In [ ]:
with open(ARCHIVO, "rb") as f:
    n = struct.unpack("<Q", f.read(8))[0]          # tamaño del encabezado
    encabezado = json.loads(f.read(n))
    pesos = np.frombuffer(f.read(), dtype="<u2")   # cada peso como entero sin signo de 16 bits

tensores = {k: v for k, v in encabezado.items() if k != "__metadata__"}
print(len(tensores), "tensores · tipos:", {v["dtype"] for v in tensores.values()})
print(f"{pesos.size:,} pesos  =  {pesos.size * 16 / 8 / 1e6:.1f} MB")

# Separar los tres campos con operaciones de bits
signo     = pesos >> 15
exponente = (pesos >> 7) & 0xFF
mantisa   = pesos & 0x7F

### 2.3 Entropía por campo

Se aplica la ecuación (2) a cada campo por separado. El "límite" es

$$\text{límite} = \frac{H(\text{signo}) + H(\text{exponente}) + H(\text{mantisa})}{16} \tag{A}$$

| Parte | Qué significa |
|---|---|
| numerador | bits que **realmente se necesitan** en promedio por peso, si cada campo se codifica de forma óptima |
| $16$ | bits que **ocupa** cada peso en `bfloat16` |
| límite | fracción del tamaño original a la que se podría reducir el archivo sin perder nada (tratando los campos como independientes) |

In [ ]:
Hs = entropia(np.bincount(signo, minlength=2))
He = entropia(np.bincount(exponente, minlength=256))
Hm = entropia(np.bincount(mantisa, minlength=128))
print(f"{'campo':10s} {'bits':>5s} {'entropía':>9s}")
for nombre, b, h in [("signo", 1, Hs), ("exponente", 8, He), ("mantisa", 7, Hm)]:
    print(f"{nombre:10s} {b:5d} {h:9.3f}")
total = Hs + He + Hm
print(f"{'total':10s} {16:5d} {total:9.3f}   ->  límite ≈ {total/16:.1%} del tamaño original")

In [ ]:
conteo_e = np.bincount(exponente, minlength=256)
usados = np.nonzero(conteo_e)[0]
print(f"Exponentes distintos usados: {len(usados)} de 256 posibles")
plt.figure(figsize=(10, 3))
plt.bar(usados - 127, conteo_e[usados] / pesos.size)
plt.xlabel("exponente (sin sesgo, potencia de 2)"); plt.ylabel("p")
plt.title("Distribución del exponente de los pesos"); plt.show()

### 2.4 ¿Y `zip`?
`zlib` (el algoritmo de `zip`) usa Lempel-Ziv + Huffman sobre **bytes**: busca secuencias de bytes repetidas.
Se prueba en los primeros 40 MB (tarda unos segundos). La **razón de compresión** es

$$\text{razón} = \frac{\text{tamaño comprimido}}{\text{tamaño original}} \tag{B}$$

donde un valor menor significa mejor compresión (0.80 = el archivo queda en 80 % de su tamaño).

In [ ]:
muestra = pesos[:20_000_000].tobytes()
razon = len(zlib.compress(muestra, 6)) / len(muestra)
print(f"zlib:            {razon:.1%} del tamaño original")
print(f"límite entrópico: {total/16:.1%}  (codificando cada campo por separado)")

### ✍️ Preguntas (respóndelas aquí; son el punto de partida del proyecto)

1. ¿Por qué el exponente tiene tan poca entropía comparado con sus 8 bits? Mira la gráfica: ¿qué dice sobre
   la magnitud típica de los pesos?
2. ¿Por qué la mantisa está casi en su máximo (7 bits)?
3. `zlib` no llega al límite. ¿Qué "no ve" `zlib` que la entropía por campo sí ve?
4. El "límite" que calculamos supone que los pesos son independientes. ¿Podría un compresor bajar todavía más?
   ¿Qué información estaríamos ignorando?

*Tus respuestas:*